# 3 — Invoice Extractor
> Paste any invoice as raw text and get back a clean, structured record — vendor, date, every line item, subtotal, tax, and total — ready to drop into your accounting system or spreadsheet.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import List
from pydantic import BaseModel, field_validator
from langchain_openai import ChatOpenAI


class LineItem(BaseModel):
    description: str
    quantity: int
    unit_price: float
    total: float


class Invoice(BaseModel):
    vendor: str
    invoice_number: str
    date: str          # ISO format: YYYY-MM-DD
    subtotal: float
    tax: float
    total_amount: float
    line_items: List[LineItem]

    @field_validator("total_amount")
    @classmethod
    def must_be_positive(cls, v: float) -> float:
        if v <= 0:
            raise ValueError("total_amount must be positive")
        return v


def run(invoice_text: str) -> Invoice:
    llm = ChatOpenAI(model="gpt-4o-mini")
    extractor = llm.with_structured_output(Invoice)
    return extractor.invoke(invoice_text)

## The scenario

A marketing agency receives an invoice from their video production vendor. The document mixes a project fee, per-day shoot rates, and a post-production retainer into a single unformatted block of text — exactly the kind of thing that takes an AP clerk five minutes to key in manually.

The agent reads the raw text and returns every field in a structured record: vendor, invoice number, date, each line item with quantity and unit price, subtotal, tax, and total. No copy-paste, no manual entry.

In [ ]:
raw_invoice = """
TAX INVOICE
From: Redframe Productions
Invoice No: RFP-2024-0341  |  Date: 22 August 2024

Client: Luminary Marketing Group

Services:
  Pre-Production & Creative Brief     1 project    @ $1,200.00      $1,200.00
  On-Location Shoot (full day)        2 days       @ $3,800.00/day  $7,600.00
  Post-Production Editing             8 hrs        @ $175.00/hr     $1,400.00
  Motion Graphics Package             1 package    @ $950.00          $950.00
  Colour Grade & Sound Mix            1 project    @ $650.00          $650.00

Subtotal ........................... $11,800.00
GST (10%) .......................... $1,180.00
TOTAL DUE .......................... $12,980.00

Payment due within 14 days. EFT details on file.
"""

invoice = run(raw_invoice)

print(f"{'='*52}")
print("  EXTRACTED INVOICE")
print(f"{'='*52}")
print(f"  Vendor:         {invoice.vendor}")
print(f"  Invoice No:     {invoice.invoice_number}")
print(f"  Date:           {invoice.date}")
print(f"  Subtotal:       ${invoice.subtotal:,.2f}")
print(f"  Tax:            ${invoice.tax:,.2f}")
print(f"  Total Due:      ${invoice.total_amount:,.2f}")
print(f"\n  Line Items ({len(invoice.line_items)}):")
for item in invoice.line_items:
    print(f"    {item.description}")
    print(f"      {item.quantity} x ${item.unit_price:,.2f} = ${item.total:,.2f}")
print(f"{'='*52}")

## Use your own data

Replace the input above with:
- Your invoice text — paste it as-is, formatting does not matter
- Any currency or tax structure (GST, VAT, sales tax — the agent adapts)

The agent will return a structured record with every line item broken out and totals validated, ready to forward to your accounting system.